In [0]:
# Databricks Notebook: 03_gold_transformations 
# Purpose: Create Gold layer - Star Schema and Business Aggregations
# Fixed column name issues

from pyspark.sql.functions import *
from pyspark.sql.window import Window
from datetime import datetime

print("="*70)
print("GOLD LAYER - STAR SCHEMA & BUSINESS INSIGHTS")
print("="*70)

# Storage account configuration
storage_account = "pharmadatalake7"
silver_base_path = f"abfss://silver@{storage_account}.dfs.core.windows.net/"
gold_base_path = f"abfss://gold@{storage_account}.dfs.core.windows.net/"

# ============================================
# SECTION 1: DIMENSION TABLES
# ============================================

print("\n" + "="*60)
print("SECTION 1: DIMENSION TABLES")
print("="*60)

# ============================================
# 1.1 Dim_Clinical_Trials 
# ============================================
print("\n1.1 Creating Dim_Clinical_Trials...")

df_ct_silver = spark.read.format("delta").load(f"{silver_base_path}clinical_trials_silver/")

# Check available columns
print("Available columns in clinical_trials_silver:")
print(df_ct_silver.columns)

dim_clinical_trials = df_ct_silver \
    .withColumn("trial_sk", monotonically_increasing_id()) \
    .withColumn("effective_date", current_date()) \
    .withColumn("is_current", lit(True)) \
    .select(
        "trial_sk",
        "trial_id",
        "trial_phase_standardized",
        "trial_status_standardized",
        # "country",  # REMOVED - column doesn't exist
        "sponsor",  # Using sponsor instead
        # "enrolled_patients",  # REMOVED - use patient_count
        "patient_count",  # Use correct column name
        "enrollment_category",
        "is_completed",
        "is_ongoing",
        "trial_year",
        "trial_month",
        "effective_date",
        "is_current"
    )

dim_clinical_trials.write.format("delta") \
    .mode("overwrite") \
    .save(f"{gold_base_path}dim_clinical_trials/")
print(f"  ✓ Dim_Clinical_Trials: {dim_clinical_trials.count()} records")

# ============================================
# 1.2 Dim_Drugs (from Sales data)
# ============================================
print("\n1.2 Creating Dim_Drugs...")

# Extract unique drugs from sales data
df_drugs = spark.read.format("delta").load(f"{silver_base_path}sales_silver/") \
    .select("drug_name") \
    .distinct() \
    .withColumn("drug_sk", monotonically_increasing_id()) \
    .withColumn("effective_date", current_date()) \
    .withColumn("is_current", lit(True))

dim_drugs = df_drugs.select(
    "drug_sk",
    "drug_name",
    "effective_date",
    "is_current"
)

dim_drugs.write.format("delta") \
    .mode("overwrite") \
    .save(f"{gold_base_path}dim_drugs/")
print(f"  ✓ Dim_Drugs: {dim_drugs.count()} records")

# ============================================
# 1.3 Dim_Physicians
# ============================================
print("\n1.3 Creating Dim_Physicians...")

df_physician_silver = spark.read.format("delta").load(f"{silver_base_path}physician_silver/")

# Check available columns
print("Available columns in physician_silver:")
print(df_physician_silver.columns)

dim_physicians = df_physician_silver \
    .withColumn("physician_sk", monotonically_increasing_id()) \
    .withColumn("effective_date", current_date()) \
    .withColumn("is_current", lit(True)) \
    .select(
        "physician_sk",
        "physician_id",
        "physician_name",
        "specialty",
        "years_experience",
        "experience_level",
        "hospital_affiliation",
        "practice_type",
        # "state",  # May not exist
        col("state") if "state" in df_physician_silver.columns else lit("Unknown").alias("state"),
        "prescription_volume_monthly",
        "most_prescribed_category",
        "patient_satisfaction_score",
        "effective_date",
        "is_current"
    )

dim_physicians.write.format("delta") \
    .mode("overwrite") \
    .save(f"{gold_base_path}dim_physicians/")
print(f"  ✓ Dim_Physicians: {dim_physicians.count()} records")

# ============================================
# 1.4 Dim_Date
# ============================================
print("\n1.4 Creating Dim_Date...")

def create_date_dimension(start_date, end_date):
    dates = spark.sql(f"""
    SELECT EXPLODE(SEQUENCE(
        TO_DATE('{start_date}'),
        TO_DATE('{end_date}'),
        INTERVAL 1 DAY
    )) AS date
    """)
    
    date_dim = dates.select(
        col("date"),
        year("date").alias("year"),
        quarter("date").alias("quarter"),
        month("date").alias("month"),
        dayofmonth("date").alias("day"),
        dayofweek("date").alias("day_of_week"),
        date_format("date", "EEEE").alias("day_name"),
        date_format("date", "MMMM").alias("month_name"),
        when(dayofweek("date").isin([1, 7]), "Weekend").otherwise("Weekday").alias("weekday_indicator")
    ).withColumn("date_sk", monotonically_increasing_id())
    
    return date_dim

dim_date = create_date_dimension("2020-01-01", "2026-12-31")

dim_date.write.format("delta") \
    .mode("overwrite") \
    .save(f"{gold_base_path}dim_date/")
print(f"  ✓ Dim_Date: {dim_date.count()} records")

# ============================================
# SECTION 2: FACT TABLES
# ============================================

print("\n" + "="*60)
print("SECTION 2: FACT TABLES")
print("="*60)

# ============================================
# 2.1 Fact_Sales
# ============================================
print("\n2.1 Creating Fact_Sales...")

df_sales_silver = spark.read.format("delta").load(f"{silver_base_path}sales_silver/")
print("Available columns in sales_silver:")
print(df_sales_silver.columns)

dim_drugs = spark.read.format("delta").load(f"{gold_base_path}dim_drugs/")
dim_date = spark.read.format("delta").load(f"{gold_base_path}dim_date/")

fact_sales = df_sales_silver \
    .join(dim_drugs.select("drug_name", "drug_sk"), "drug_name", "left") \
    .join(dim_date.select("date", "date_sk"), df_sales_silver["sales_date"] == dim_date["date"], "left") \
    .select(
        "sale_id",
        "drug_sk",
        "date_sk",
        "sales_channel_standardized",
        "units_sold",
        "unit_price_usd",
        "total_revenue_usd",
        "region",
        "country" if "country" in df_sales_silver.columns else lit("Unknown").alias("country"),
        "sales_rep_id" if "sales_rep_id" in df_sales_silver.columns else lit("Unknown").alias("sales_rep_id"),
        "sales_year",
        "sales_month",
        "sales_quarter",
        "is_high_value",
        "revenue_category"
    )

fact_sales.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(f"{gold_base_path}fact_sales/")
print(f"  ✓ Fact_Sales: {fact_sales.count()} records")

# ============================================
# 2.2 Fact_Clinical_Trials 
# ============================================
print("\n2.2 Creating Fact_Clinical_Trials...")

df_ct_silver = spark.read.format("delta").load(f"{silver_base_path}clinical_trials_silver/")
dim_clinical_trials = spark.read.format("delta").load(f"{gold_base_path}dim_clinical_trials/")
dim_date = spark.read.format("delta").load(f"{gold_base_path}dim_date/")

# Check available columns
print("Available columns in clinical_trials_silver for fact:")
print(df_ct_silver.columns)

fact_clinical_trials = df_ct_silver \
    .join(dim_clinical_trials.select("trial_id", "trial_sk"), "trial_id", "left") \
    .join(dim_date.select("date", "date_sk"), df_ct_silver["start_date"] == dim_date["date"], "left") \
    .select(
        "trial_id",
        "trial_sk",
        "date_sk",
        "drug_name",
        "condition_treated",
        "patient_count",
        "efficacy_rate",
        "placebo_efficacy" if "placebo_efficacy" in df_ct_silver.columns else lit(0.0).alias("placebo_efficacy"),
        "adverse_events_count",
        "serious_adverse_events" if "serious_adverse_events" in df_ct_silver.columns else lit(0).alias("serious_adverse_events"),
        "budget_millions" if "budget_millions" in df_ct_silver.columns else lit(0.0).alias("budget_millions"),
        "completion_rate" if "completion_rate" in df_ct_silver.columns else lit(0.0).alias("completion_rate"),
        "data_quality_score" if "data_quality_score" in df_ct_silver.columns else lit(0.0).alias("data_quality_score")
    )

fact_clinical_trials.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(f"{gold_base_path}fact_clinical_trials/")
print(f"  ✓ Fact_Clinical_Trials: {fact_clinical_trials.count()} records")

# ============================================
# 2.3 Fact_Prescriptions 
# ============================================
print("\n2.3 Creating Fact_Prescriptions...")

df_rx_silver = spark.read.format("delta").load(f"{silver_base_path}prescriptions_silver/")
dim_physicians = spark.read.format("delta").load(f"{gold_base_path}dim_physicians/")
dim_date = spark.read.format("delta").load(f"{gold_base_path}dim_date/")

# Print columns for debugging
print("Columns in prescriptions_silver:")
print(df_rx_silver.columns)
print("Columns in dim_physicians:")
print(dim_physicians.columns)

# Check if prescribing_doctor exists, if not use physician_name
join_column = "prescribing_doctor" if "prescribing_doctor" in df_rx_silver.columns else "physician_name"

fact_prescriptions = df_rx_silver \
    .join(dim_physicians.select("physician_name", "physician_sk"), 
          df_rx_silver[join_column] == dim_physicians["physician_name"], 
          "left") \
    .join(dim_date.select("date", "date_sk"), 
          df_rx_silver["prescription_date"] == dim_date["date"], "left") \
    .select(
        "prescription_id",
        "physician_sk",
        "date_sk",
        "patient_id",
        "drug_name",
        "dosage_mg",
        "frequency_standardized",
        "duration_days",
        "duration_category",
        "refill_count",
        "adherence_score",
        "adherence_level",
        "copay_amount" if "copay_amount" in df_rx_silver.columns else lit(0.0).alias("copay_amount"),
        "insurance_provider"
    )

fact_prescriptions.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(f"{gold_base_path}fact_prescriptions/")
print(f"  ✓ Fact_Prescriptions: {fact_prescriptions.count()} records")

# ============================================
# 2.4 Fact_Inventory
# ============================================
print("\n2.4 Creating Fact_Inventory...")

df_inv_silver = spark.read.format("delta").load(f"{silver_base_path}inventory_silver/")
print("Available columns in inventory_silver:")
print(df_inv_silver.columns)

fact_inventory = df_inv_silver \
    .select(
        "inventory_id",
        "drug_name",
        "batch_number",
        "warehouse_location",
        "drug_type_standardized",
        "quantity_in_stock",
        "reorder_level" if "reorder_level" in df_inv_silver.columns else lit(0).alias("reorder_level"),
        "safety_stock" if "safety_stock" in df_inv_silver.columns else lit(0).alias("safety_stock"),
        "reorder_quantity" if "reorder_quantity" in df_inv_silver.columns else lit(0).alias("reorder_quantity"),
        "supplier",
        "lead_time_days" if "lead_time_days" in df_inv_silver.columns else lit(0).alias("lead_time_days"),
        "storage_cost_per_unit" if "storage_cost_per_unit" in df_inv_silver.columns else lit(0.0).alias("storage_cost_per_unit"),
        "expiry_date",
        "manufacturing_date" if "manufacturing_date" in df_inv_silver.columns else lit("2020-01-01").alias("manufacturing_date"),
        "batch_quality_score" if "batch_quality_score" in df_inv_silver.columns else lit(0.0).alias("batch_quality_score"),
        "shipping_method" if "shipping_method" in df_inv_silver.columns else lit("Unknown").alias("shipping_method"),
        "holding_days" if "holding_days" in df_inv_silver.columns else lit(0).alias("holding_days"),
        "stockout_risk_standardized",
        "days_to_expiry",
        "expiry_risk",
        "stock_ratio" if "stock_ratio" in df_inv_silver.columns else lit(0.0).alias("stock_ratio"),
        "stock_status"
    )

fact_inventory.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(f"{gold_base_path}fact_inventory/")
print(f"  ✓ Fact_Inventory: {fact_inventory.count()} records")

# ============================================
# 2.5 Fact_Marketing
# ============================================
print("\n2.5 Creating Fact_Marketing...")

df_marketing_silver = spark.read.format("delta").load(f"{silver_base_path}marketing_silver/")
print("Available columns in marketing_silver:")
print(df_marketing_silver.columns)

dim_date = spark.read.format("delta").load(f"{gold_base_path}dim_date/")

fact_marketing = df_marketing_silver \
    .join(dim_date.select("date", "date_sk"), df_marketing_silver["campaign_start"] == dim_date["date"], "left") \
    .select(
        "campaign_id",
        "date_sk",
        "drug_name",
        "campaign_name",
        "channel",
        "target_audience",
        "budget_usd",
        "spent_usd",
        "impressions",
        "clicks",
        "conversion_rate",
        "reach_count" if "reach_count" in df_marketing_silver.columns else lit(0).alias("reach_count"),
        "engagement_rate" if "engagement_rate" in df_marketing_silver.columns else lit(0.0).alias("engagement_rate"),
        "roi_percent",
        "roi_category",
        "drug_type" if "drug_type" in df_marketing_silver.columns else lit("Unknown").alias("drug_type"),
        "seasonal" if "seasonal" in df_marketing_silver.columns else lit("No").alias("seasonal")
    )

fact_marketing.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(f"{gold_base_path}fact_marketing/")
print(f"  ✓ Fact_Marketing: {fact_marketing.count()} records")

print("\n" + "="*60)
print("✅ GOLD LAYER COMPLETED - All fact & dimension tables saved")
print("="*60)


In [0]:
%sql
-- ============================================
-- CREATE ALL GOLD TABLES IN UNITY CATALOG
-- ============================================

USE CATALOG pharma_catalog_prod;
USE SCHEMA gold;

-- Dimension Tables
CREATE TABLE IF NOT EXISTS pharma_catalog_prod.gold.dim_clinical_trials
USING DELTA
LOCATION 'abfss://gold@pharmadatalake7.dfs.core.windows.net/dim_clinical_trials/';

CREATE TABLE IF NOT EXISTS pharma_catalog_prod.gold.dim_drugs
USING DELTA
LOCATION 'abfss://gold@pharmadatalake7.dfs.core.windows.net/dim_drugs/';

CREATE TABLE IF NOT EXISTS pharma_catalog_prod.gold.dim_physicians
USING DELTA
LOCATION 'abfss://gold@pharmadatalake7.dfs.core.windows.net/dim_physicians/';

CREATE TABLE IF NOT EXISTS pharma_catalog_prod.gold.dim_date
USING DELTA
LOCATION 'abfss://gold@pharmadatalake7.dfs.core.windows.net/dim_date/';

-- Fact Tables
CREATE TABLE IF NOT EXISTS pharma_catalog_prod.gold.fact_sales
USING DELTA
LOCATION 'abfss://gold@pharmadatalake7.dfs.core.windows.net/fact_sales/';

CREATE TABLE IF NOT EXISTS pharma_catalog_prod.gold.fact_clinical_trials
USING DELTA
LOCATION 'abfss://gold@pharmadatalake7.dfs.core.windows.net/fact_clinical_trials/';

CREATE TABLE IF NOT EXISTS pharma_catalog_prod.gold.fact_prescriptions
USING DELTA
LOCATION 'abfss://gold@pharmadatalake7.dfs.core.windows.net/fact_prescriptions/';

CREATE TABLE IF NOT EXISTS pharma_catalog_prod.gold.fact_inventory
USING DELTA
LOCATION 'abfss://gold@pharmadatalake7.dfs.core.windows.net/fact_inventory/';

CREATE TABLE IF NOT EXISTS pharma_catalog_prod.gold.fact_marketing
USING DELTA
LOCATION 'abfss://gold@pharmadatalake7.dfs.core.windows.net/fact_marketing/';

-- Verify all tables
SHOW TABLES IN pharma_catalog_prod.gold;
